In [ ]:
# ============================================================================
# CourseSense — FULL PIPELINE IN ONE CELL
# Dataset -> Preprocessing -> Feature Engineering -> GRU -> Recommendations
# ============================================================================
# This condenses everything from build_dataset_and_preprocess.py,
# feature_engineering.py, gru_optuna_tuning.py (using the hyperparameters
# that search already found best, rather than re-running the full search),
# and the fixed recommend.py into one linear script, for reference.
#
# NOTE: this uses the Optuna-tuned hyperparameters we already found
# (embed_dim=32, gru_units=48, dropout=0.101, l2reg=0.00077, lr=0.00467) —
# it does NOT re-run the Optuna search itself. If you want to search again
# from scratch, use the separate gru_optuna_tuning.py script; that search
# takes far longer than makes sense for a single reference cell.
# ============================================================================

# !pip install nltk -q

import re
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)
import nltk

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


# ============================================================================
# STEP 1 — DOWNLOAD & SAMPLE THE DATASET
# ============================================================================
# Each row = one student's attempt at one skill, in chronological order.
# We sample whole students (never split a sequence) to land in the 9k-20k
# row range while keeping each student's full history intact.

DATA_URL = "https://raw.githubusercontent.com/GaoSida/DKT/master/Assistments/skill_builder_data.csv"
print("Downloading dataset...")
raw = pd.read_csv(DATA_URL, encoding="ISO-8859-1", low_memory=False)

KEEP_COLS = ["order_id", "assignment_id", "user_id", "problem_id", "skill_id", "skill_name",
             "correct", "attempt_count", "hint_count", "hint_total", "ms_first_response", "overlap_time"]
raw = raw[KEEP_COLS].dropna(subset=["skill_name", "skill_id"]).reset_index(drop=True)

counts = raw.groupby("user_id").size()
eligible_students = counts[(counts >= 15) & (counts <= 120)].index.to_numpy().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(eligible_students)

TARGET_ROWS = 15000
chosen, total = [], 0
for sid in eligible_students:
    n = counts[sid]
    if total + n > TARGET_ROWS + 1000:
        continue
    chosen.append(sid)
    total += n
    if total >= TARGET_ROWS:
        break

df = raw[raw["user_id"].isin(chosen)].copy()
df = df.sort_values(["user_id", "order_id"]).reset_index(drop=True)
print(f"Sampled: {df.shape[0]} rows, {df['user_id'].nunique()} students, "
      f"{df['skill_name'].nunique()} raw skills")


# ============================================================================
# STEP 2 — TEXT PREPROCESSING (on skill_name)
# ============================================================================
# Lowercase -> strip non-alphabetic -> tokenize -> remove stopwords ->
# lemmatize. This is the NLP step: it collapses near-duplicate skill labels
# (e.g. plural/singular forms) into a consistent vocabulary used everywhere
# downstream (the model's skill embedding, side features, and recommender).

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_skill_name(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t, pos="n") for t in tokens]
    return " ".join(tokens) if tokens else text

df["clean_skill_name"] = df["skill_name"].apply(clean_skill_name)
print(f"Cleaned skill vocabulary: {df['clean_skill_name'].nunique()} unique skills")


# ============================================================================
# STEP 3 — FEATURE ENGINEERING (PFA-style, leakage-checked)
# ============================================================================
# NOTE on features deliberately EXCLUDED: attempt_count and hint_count/
# hint_total are NOT used. In this dataset `correct` means "right on the
# FIRST attempt", so attempt_count > 1 is near-tautological with correct=0
# (verified: ~79% correct at attempt_count==1, <2% for any higher value).
# Including them would make prediction trivially circular.

df["opportunity"] = df.groupby(["user_id", "clean_skill_name"]).cumcount() + 1
df["_cum_correct"] = df.groupby(["user_id", "clean_skill_name"])["correct"].cumsum()
df["prior_success_count"] = df["_cum_correct"] - df["correct"]
df["prior_failure_count"] = (df["opportunity"] - 1) - df["prior_success_count"]

skill_sum = df.groupby("clean_skill_name")["correct"].transform("sum")
skill_cnt = df.groupby("clean_skill_name")["correct"].transform("count")
df["skill_difficulty"] = (skill_sum - df["correct"]) / (skill_cnt - 1)  # leave-one-out

df["_cum_correct_student"] = df.groupby("user_id")["correct"].cumsum()
df["student_interaction_number"] = df.groupby("user_id").cumcount() + 1
df["prior_student_correct"] = df["_cum_correct_student"] - df["correct"]
df["student_rolling_accuracy"] = df["prior_student_correct"] / (df["student_interaction_number"] - 1)
df["student_rolling_accuracy"] = df["student_rolling_accuracy"].fillna(df["correct"].mean())

df["log_ms_first_response"] = np.log1p(df["ms_first_response"].clip(lower=0))
df["log_overlap_time"] = np.log1p(df["overlap_time"].clip(lower=0))

df = df.drop(columns=["_cum_correct", "_cum_correct_student", "prior_student_correct",
                       "attempt_count", "hint_count", "hint_total"])

# skill_vocab: clean_skill_name -> 0..num_skills-1 (the model's canonical vocabulary)
skills = sorted(df["clean_skill_name"].unique())
skill_to_idx = {s: i for i, s in enumerate(skills)}
idx_to_skill = {i: s for s, i in skill_to_idx.items()}
num_skills = len(skills)
df["skill_idx"] = df["clean_skill_name"].map(skill_to_idx)
df["interaction_id"] = df["skill_idx"] + df["correct"] * num_skills  # DKT-style encoding

engineered_features = df.copy()  # this is the equivalent of engineered_features.csv


# ============================================================================
# STEP 4 — BUILD PADDED SEQUENCES FOR THE GRU
# ============================================================================
sequences = df.groupby("user_id").agg(
    skill_seq=("skill_idx", list), correct_seq=("correct", list), interaction_seq=("interaction_id", list),
).reset_index()
sequences["seq_len"] = sequences["skill_seq"].apply(len)
max_len = sequences["seq_len"].max()
sorted_user_ids = np.sort(df["user_id"].unique())
n_students = len(sorted_user_ids)

PAD_TOKEN = 2 * num_skills
START_TOKEN = 2 * num_skills + 1

skill_arr = np.full((n_students, max_len), -1, dtype=np.int32)
correct_arr = np.full((n_students, max_len), -1, dtype=np.int32)
interaction_arr = np.full((n_students, max_len), PAD_TOKEN, dtype=np.int32)
seq_lengths = np.zeros(n_students, dtype=np.int32)
for i, row in sequences.iterrows():
    L = row["seq_len"]
    skill_arr[i, :L] = row["skill_seq"]
    correct_arr[i, :L] = row["correct_seq"]
    interaction_arr[i, :L] = row["interaction_seq"]
    seq_lengths[i] = L

# Shifted interaction input: position t's input = the PREVIOUS interaction
X_shifted_all = np.full((n_students, max_len), PAD_TOKEN, dtype=np.int32)
X_shifted_all[:, 0] = START_TOKEN
X_shifted_all[:, 1:] = interaction_arr[:, :-1]

valid_mask_all = np.zeros((n_students, max_len), dtype=np.float32)
for i, L in enumerate(seq_lengths):
    valid_mask_all[i, :L] = 1.0
y_all = np.where(correct_arr == -1, 0, correct_arr).astype(np.float32)

SIDE_FEATS_BASE = ["opportunity", "prior_success_count", "prior_failure_count",
                    "student_rolling_accuracy", "log_ms_first_response", "log_overlap_time"]
grouped = df.groupby("user_id")
per_student_rows = {uid: grouped.get_group(uid) for uid in sorted_user_ids}
base_side_all = np.zeros((n_students, max_len, len(SIDE_FEATS_BASE)), dtype=np.float32)
for i, uid in enumerate(sorted_user_ids):
    g = per_student_rows[uid]
    base_side_all[i, :len(g), :] = g[SIDE_FEATS_BASE].values

print(f"Sequences built: {n_students} students x {max_len} max timesteps, {num_skills} skills")


# ============================================================================
# STEP 5 — TRAIN/TEST SPLIT (by student, never split a sequence)
# ============================================================================
rng2 = np.random.default_rng(SEED)
students_for_split = df["user_id"].unique()
rng2.shuffle(students_for_split)
n_test = int(len(students_for_split) * 0.2)
test_students = set(students_for_split[:n_test])
train_students = set(students_for_split[n_test:])
train_pool_idx = np.array([i for i, uid in enumerate(sorted_user_ids) if uid in train_students])
test_idx = np.array([i for i, uid in enumerate(sorted_user_ids) if uid in test_students])

def build_skill_difficulty_column(train_idx_local, target_idx_local):
    """Recompute skill_difficulty from TRAIN students only, apply to target -- avoids leakage."""
    train_uids = set(sorted_user_ids[i] for i in train_idx_local)
    train_rows = df[df["user_id"].isin(train_uids)]
    diff_map = train_rows.groupby("clean_skill_name")["correct"].mean()
    global_mean = train_rows["correct"].mean()
    col = np.zeros((len(target_idx_local), max_len, 1), dtype=np.float32)
    for j, i in enumerate(target_idx_local):
        g = per_student_rows[sorted_user_ids[i]]
        vals = g["clean_skill_name"].map(diff_map).fillna(global_mean).values
        col[j, :len(g), 0] = vals
    return col

diff_train = build_skill_difficulty_column(train_pool_idx, train_pool_idx)
diff_test = build_skill_difficulty_column(train_pool_idx, test_idx)
side_train = np.concatenate([base_side_all[train_pool_idx], diff_train], axis=-1)
side_test = np.concatenate([base_side_all[test_idx], diff_test], axis=-1)


# ============================================================================
# STEP 6 — BUILD & TRAIN THE GRU (using the Optuna-tuned hyperparameters)
# ============================================================================
BEST_PARAMS = {"embed_dim": 32, "gru_units": 48, "dropout": 0.100819022070179,
                "l2reg": 0.0007689178899002101, "lr": 0.00466851730856359}
VOCAB_SIZE = 2 * num_skills + 2

inter_in = keras.Input(shape=(max_len,), dtype="int32")
side_in = keras.Input(shape=(max_len, side_train.shape[-1]), dtype="float32")
emb = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=BEST_PARAMS["embed_dim"])(inter_in)
merged = layers.Concatenate()([emb, side_in])
gru = layers.GRU(BEST_PARAMS["gru_units"], return_sequences=True, dropout=BEST_PARAMS["dropout"],
                  kernel_regularizer=keras.regularizers.l2(BEST_PARAMS["l2reg"]))(merged)
out = layers.Dense(1, activation="sigmoid")(gru)  # shape (batch, T, 1)
model = keras.Model([inter_in, side_in], out)
model.compile(optimizer=keras.optimizers.Adam(BEST_PARAMS["lr"]), loss="binary_crossentropy")

early_stop = keras.callbacks.EarlyStopping(monitor="loss", patience=8, restore_best_weights=True)
model.fit(
    [X_shifted_all[train_pool_idx], side_train], y_all[train_pool_idx][..., None],
    sample_weight=valid_mask_all[train_pool_idx][..., None],
    epochs=60, batch_size=16, callbacks=[early_stop], verbose=2,
)


# ============================================================================
# STEP 7 — EVALUATE ON THE HELD-OUT TEST STUDENTS
# ============================================================================
def evaluate(idx_set, side_feats, label):
    preds = model.predict([X_shifted_all[idx_set], side_feats], verbose=0)[..., 0]
    mask = valid_mask_all[idx_set] > 0
    y_t, y_p = y_all[idx_set][mask], preds[mask]
    y_pred_bin = (y_p >= 0.5).astype(int)
    print(f"\n=== {label} ===")
    print(f"Accuracy: {accuracy_score(y_t, y_pred_bin):.4f} | "
          f"Precision: {precision_score(y_t, y_pred_bin):.4f} | "
          f"Recall: {recall_score(y_t, y_pred_bin):.4f} | "
          f"F1: {f1_score(y_t, y_pred_bin):.4f} | "
          f"ROC-AUC: {roc_auc_score(y_t, y_p):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_t, y_pred_bin))

evaluate(test_idx, side_test, "HELD-OUT TEST performance")


# ============================================================================
# STEP 8 — THE RECOMMENDATION ENGINE
# ============================================================================
# For a given student, ask the model a "what if" question for every skill:
# "if this student attempted skill X next, what's their predicted success
# probability?" — built from their real history plus one hypothetical step.
# Skills with fewer than MIN_SKILL_OCCURRENCES total examples are excluded,
# since their skill_difficulty collapses to an uninformative default with
# too little data (this was the root cause of a real bug we hit earlier).

MIN_SKILL_OCCURRENCES = 20
skill_difficulty_map = df.groupby("clean_skill_name")["correct"].mean().to_dict()  # full-data, for deployed use
skill_occurrence_counts = df["clean_skill_name"].value_counts()

def recommend_next_skills(student, top_k=5, strategy="remediate"):
    student_row_idx = student if isinstance(student, (int, np.integer)) and 0 <= student < n_students else int(np.where(sorted_user_ids == student)[0][0])
    L = int(seq_lengths[student_row_idx])
    hist_skills = skill_arr[student_row_idx, :L]
    hist_correct = correct_arr[student_row_idx, :L]
    hist_interactions = interaction_arr[student_row_idx, :L]

    opp_count = {s: 0 for s in range(num_skills)}
    succ_count = {s: 0 for s in range(num_skills)}
    fail_count = {s: 0 for s in range(num_skills)}
    for s, c in zip(hist_skills, hist_correct):
        opp_count[s] += 1
        succ_count[s] += 1 if c == 1 else 0
        fail_count[s] += 1 if c == 0 else 0
    rolling_acc = hist_correct.sum() / L

    student_uid = sorted_user_ids[student_row_idx]
    student_rows = df[df["user_id"] == student_uid]
    avg_log_rt = student_rows["log_ms_first_response"].mean()
    avg_log_ot = student_rows["log_overlap_time"].mean()
    hist_side = student_rows[SIDE_FEATS_BASE].values
    hist_diff = np.array([skill_difficulty_map[idx_to_skill[s]] for s in hist_skills])

    candidates = [i for i in range(num_skills)
                  if skill_occurrence_counts.get(idx_to_skill[i], 0) >= MIN_SKILL_OCCURRENCES]
    excluded = [idx_to_skill[i] for i in range(num_skills) if i not in candidates]

    X_batch = np.full((len(candidates), max_len), PAD_TOKEN, dtype=np.int32)
    side_batch = np.zeros((len(candidates), max_len, len(SIDE_FEATS_BASE) + 1), dtype=np.float32)
    for row, cand in enumerate(candidates):
        X_batch[row, 0] = START_TOKEN
        X_batch[row, 1:L] = hist_interactions[:L - 1]
        X_batch[row, L] = hist_interactions[L - 1]
        side_batch[row, :L, :len(SIDE_FEATS_BASE)] = hist_side
        side_batch[row, :L, len(SIDE_FEATS_BASE)] = hist_diff
        side_batch[row, L, 0] = opp_count[cand] + 1
        side_batch[row, L, 1] = succ_count[cand]
        side_batch[row, L, 2] = fail_count[cand]
        side_batch[row, L, 3] = rolling_acc
        side_batch[row, L, 4] = avg_log_rt
        side_batch[row, L, 5] = avg_log_ot
        side_batch[row, L, 6] = skill_difficulty_map[idx_to_skill[cand]]

    preds = model.predict([X_batch, side_batch], verbose=0)[:, L, 0]
    results = pd.DataFrame({
        "skill": [idx_to_skill[c] for c in candidates],
        "predicted_success_prob": preds,
        "times_practiced_before": [opp_count[c] for c in candidates],
    }).sort_values("predicted_success_prob").reset_index(drop=True)

    if strategy == "remediate":
        return results.head(top_k), excluded
    elif strategy == "zpd":
        results["dist"] = (results["predicted_success_prob"] - 0.60).abs()
        return results.sort_values("dist").head(top_k).drop(columns="dist"), excluded


# ============================================================================
# STEP 9 — DEMO
# ============================================================================
for demo_idx in [0, 5]:
    uid = sorted_user_ids[demo_idx]
    print(f"\n{'='*60}\nStudent user_id={uid} ({seq_lengths[demo_idx]} interactions so far)\n{'='*60}")
    rec, excl = recommend_next_skills(demo_idx, top_k=5, strategy="remediate")
    print("\n-- Remediation --")
    print(rec.to_string(index=False))
    rec, excl = recommend_next_skills(demo_idx, top_k=5, strategy="zpd")
    print("\n-- ZPD --")
    print(rec.to_string(index=False))